# FINCORP DistilRoBERTa — V1–V5

Runnable severity-classification experiment notebook. Runs on Google Colab with GPU. Historical metrics are saved separately in the repository; fresh scores depend on the runtime and split.

## 1. Setup
Enable GPU (Runtime → Change runtime type), then run setup.

In [ ]:
%pip -q install "transformers==4.57.6" pandas numpy scikit-learn safetensors
import subprocess,sys,torch
from pathlib import Path
repo=Path('/content/FINCORP_Reproduction')
if not repo.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/Mohame-AlShazly/FINCORP_Reproduction.git',str(repo)],check=True)
print('PyTorch:',torch.__version__,'GPU:',torch.cuda.is_available())
if not torch.cuda.is_available(): print('WARNING: Enable GPU before training')


## 2. Training and evaluation functions

In [ ]:
from __future__ import annotations
import os,json,random,re,time,zipfile
from pathlib import Path
import numpy as np,pandas as pd,torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix,classification_report
from torch.utils.data import Dataset,DataLoader
from transformers import AutoTokenizer,AutoModelForSequenceClassification,get_linear_schedule_with_warmup

ROOT=Path('/content/FINCORP_Reproduction')
DATA=ROOT/'FINCORP_Improvment Experiments/data/FINCORP.csv'
EXTERNAL=ROOT/'FINCORP_Improvment Experiments/FINCORP_Jin_Aletras_External_Dataset_Ready/jin_aletras_complaints_only.csv'
OUT=Path('/content/fincorp_neural_outputs')
MODEL='distilbert/distilroberta-base'
LABELS=['Accusation','Blame','Disapproval','No explicit reproach','Non-Complaint']
SEED=40;MAX_LENGTH=128;BATCH=8;ACCUM=2;LR=2e-5;EPOCHS=5
V1_WEIGHTS=[2.0906252495830744,0.8349216635698001,1.0054745818202278,0.6276499874410092,0.4413285175858891]

def seed_all():
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)

def norm(s):
    return re.sub(r'\s+',' ',re.sub(r'https?://\S+|www\.\S+',' URL ',str(s))).strip().lower()

def load_data():
    df=pd.read_csv(DATA)
    assert len(df)==6281
    idx=np.arange(len(df))
    train_val,test=train_test_split(idx,test_size=.30,random_state=SEED,stratify=df['Severity level'])
    train,val=train_test_split(train_val,test_size=1/7,random_state=SEED,stratify=df.iloc[train_val]['Severity level'])
    assert (len(train),len(val),len(test))==(3768,628,1885)
    def subset(ids):
        return pd.DataFrame({'text':df.iloc[ids]['Complaint/ Opinion'].astype(str).to_numpy(),'label':df.iloc[ids]['Severity level'].astype(str).to_numpy(),'row_index':ids})
    return df,subset(train),subset(val),test

class Items(Dataset):
    def __init__(self,rows,tok,labels):
        self.encoded=tok(rows.text.tolist(),truncation=True,max_length=MAX_LENGTH)
        mapping={s:i for i,s in enumerate(labels)}
        self.y=[mapping[x] for x in rows.label]
    def __len__(self):return len(self.y)
    def __getitem__(self,i):return {k:v[i] for k,v in self.encoded.items()}|{'labels':self.y[i]}

def make_loader(rows,tok,labels,shuffle=False):
    def collate(batch):
        encoded=tok.pad([{k:v for k,v in b.items() if k!='labels'} for b in batch],padding=True,return_tensors='pt')
        encoded['labels']=torch.tensor([b['labels'] for b in batch],dtype=torch.long)
        return encoded
    return DataLoader(Items(rows,tok,labels),batch_size=BATCH,shuffle=shuffle,collate_fn=collate,generator=torch.Generator().manual_seed(SEED))

@torch.no_grad()
def predict(model,dl,device):
    model.eval();y=[];probs=[]
    for batch in dl:
        y.extend(batch.pop('labels').tolist())
        x={k:v.to(device) for k,v in batch.items()}
        probs.append(torch.softmax(model(**x).logits.float().cpu(),dim=-1).numpy())
    return np.array(y),np.concatenate(probs)

def score(y,pred,labels):
    return {'accuracy':float(accuracy_score(y,pred)),'macro_f1':float(f1_score(y,pred,average='macro',labels=list(range(len(labels))),zero_division=0))}

def save_predictions(folder,rows,y,probs,labels,pred=None):
    folder.mkdir(parents=True,exist_ok=True)
    if pred is None:pred=probs.argmax(1)
    result=score(y,pred,labels)
    (folder/'metrics.json').write_text(json.dumps(result,indent=2))
    pd.DataFrame({'row_index':rows.row_index.to_list(),'true_label':rows.label.to_list(),'prediction':[labels[i] for i in pred],**{f'p_{s}':probs[:,i] for i,s in enumerate(labels)}}).to_csv(folder/'validation_predictions.csv',index=False)
    pd.DataFrame(confusion_matrix(y,pred,labels=list(range(len(labels)))),index=labels,columns=labels).to_csv(folder/'confusion.csv')
    (folder/'classification_report.txt').write_text(classification_report(y,pred,labels=list(range(len(labels))),target_names=labels,zero_division=0))
    return result

def train(name,training,validation,labels,epochs=EPOCHS,weights=None,starting=None):
    seed_all()
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tok=AutoTokenizer.from_pretrained(MODEL)
    model=AutoModelForSequenceClassification.from_pretrained(starting or MODEL,num_labels=len(labels),id2label={i:v for i,v in enumerate(labels)},label2id={v:i for i,v in enumerate(labels)},ignore_mismatched_sizes=starting is not None).to(device)
    tr=make_loader(training,tok,labels,True);va=make_loader(validation,tok,labels,False)
    opt=torch.optim.AdamW(model.parameters(),lr=LR)
    total=epochs*((len(tr)+ACCUM-1)//ACCUM)
    scheduler=get_linear_schedule_with_warmup(opt,num_warmup_steps=0,num_training_steps=total)
    weight=torch.tensor(weights,dtype=torch.float32,device=device) if weights is not None else None
    folder=OUT/name;folder.mkdir(parents=True,exist_ok=True)
    best=-1;history=[]
    for epoch in range(epochs):
        model.train();opt.zero_grad(set_to_none=True);losses=[];start=time.time()
        for i,batch in enumerate(tr):
            targets=batch.pop('labels').to(device)
            x={k:v.to(device) for k,v in batch.items()}
            loss=torch.nn.functional.cross_entropy(model(**x).logits,targets,weight=weight)
            losses.append(float(loss.detach().cpu()))
            size=min(ACCUM,len(tr)-(i//ACCUM)*ACCUM)
            (loss/size).backward()
            if (i+1)%ACCUM==0 or i+1==len(tr):
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                opt.step();scheduler.step();opt.zero_grad(set_to_none=True)
        y,probs=predict(model,va,device)
        result=score(y,probs.argmax(1),labels)
        item={'epoch':epoch+1,'train_loss':float(np.mean(losses)),**result,'seconds':round(time.time()-start,1)}
        history.append(item);pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
        print(name,'epoch',epoch+1,'/',epochs,item,flush=True)
        if result['macro_f1']>best:
            best=result['macro_f1']
            model.save_pretrained(folder/'best_model',safe_serialization=True)
            tok.save_pretrained(folder/'best_model')
            save_predictions(folder,validation,y,probs,labels)
            (folder/'best_epoch.json').write_text(json.dumps(item,indent=2))
    return folder/'best_model'

def load_external(fincorp):
    ext=pd.read_csv(EXTERNAL).rename(columns={'Severity level':'label'})
    ext=ext[ext.label.isin(LABELS[:4])].dropna(subset=['text','label']).copy().reset_index(drop=True)
    original=set(fincorp['Complaint/ Opinion'].map(norm))
    ext=ext[~ext.text.map(norm).isin(original)].copy().reset_index(drop=True)
    ext['row_index']=np.arange(len(ext))
    print('External rows after exact-text exclusion:',len(ext))
    ix=np.arange(len(ext))
    tr,va=train_test_split(ix,test_size=.10,random_state=SEED,stratify=ext.label)
    return ext.iloc[tr].reset_index(drop=True),ext.iloc[va].reset_index(drop=True)

def run(name):
    if name not in ('V1','V2','V3','V4','V5'):raise ValueError('Choose V1, V2, V3, V4, V5')
    seed_all()
    source,tr,va,test=load_data()
    print('FINCORP split:',len(tr),len(va),len(test),'test reserved, not evaluated')
    print('Training labels:',tr.label.value_counts().to_dict())
    if name=='V1':return train(name,tr,va,LABELS,weights=V1_WEIGHTS)
    if name=='V2':return train(name,tr,va,LABELS)
    if name=='V3':
        path=OUT/'V2'/'validation_predictions.csv'
        if not path.is_file():raise FileNotFoundError('Run V2 first')
        pred=pd.read_csv(path)
        if pred.row_index.to_list()!=va.row_index.to_list():raise ValueError('Different validation split/order')
        p=pred[[f'p_{x}' for x in LABELS]].to_numpy()
        y=np.array([LABELS.index(v) for v in va.label])
        guess=np.where(p[:,0]>=0.05,0,p.argmax(1))
        result=save_predictions(OUT/'V3',va,y,p,LABELS,guess)
        print('V3 result:',result)
        return result
    if name=='V4':return train(name,tr,va,LABELS,weights=[2.5,1,1,1,1])
    ext_tr,ext_va=load_external(source)
    checkpoint=train('V5_external_stage',ext_tr,ext_va,LABELS[:4],epochs=3)
    return train('V5',tr,va,LABELS,starting=str(checkpoint))


## V2: unweighted baseline

In [ ]:
run('V2')

## V3: threshold adjustment to V2 predictions

In [ ]:
run('V3')

## V1: frequency-weighted cross entropy

In [ ]:
run('V1')

## V4: Accusation weight 2.5

In [ ]:
run('V4')

## V5: external-to-FINCORP transfer

In [ ]:
run('V5')

## Save metrics and predictions
Download summary archive. The trained checkpoints remain in `/content/fincorp_neural_outputs/` until you save them to persistent storage.

In [ ]:
import zipfile,json
from pathlib import Path
out=Path('/content/fincorp_neural_outputs')
with zipfile.ZipFile('/content/FINCORP_DistilRoBERTa_results.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob('*'):
        if p.is_file() and 'best_model' not in p.parts:
            z.write(p,p.relative_to(out))
print('Saved /content/FINCORP_DistilRoBERTa_results.zip')
from google.colab import files
files.download('/content/FINCORP_DistilRoBERTa_results.zip')